# NEXUS - RED SOCIAL SEGURA

Notebook organizado para conectar Azure SQL, procesar los datos, actualizar el Data Warehouse, calcular analítica y exponer resultados a PHP mediante FastAPI.

Ejecuta las celdas en orden. Las credenciales se solicitan de forma interactiva; no guardes contraseñas ni tokens en el notebook compartido.


## 0. Instalar dependencias y ODBC Driver 18

In [ ]:
# ============================================================
# NEXUS - DEPENDENCIAS Y DRIVER SQL SERVER PARA COLAB
# ============================================================
!curl -fsSL https://packages.microsoft.com/keys/microsoft.asc | gpg --dearmor -o /usr/share/keyrings/microsoft-prod.gpg
!curl -fsSL https://packages.microsoft.com/config/ubuntu/22.04/prod.list | tee /etc/apt/sources.list.d/mssql-release.list
!apt-get update -qq
!ACCEPT_EULA=Y apt-get install -y msodbcsql18 unixodbc-dev
!pip -q install --upgrade mssql-python pandas numpy requests fastapi uvicorn nest_asyncio scikit-learn

from mssql_python import connect
print('Dependencias y controlador SQL Server preparados.')


## 1. Configuración de Azure, token y conexiones

In [ ]:
# ============================================================
# NEXUS BI - CONFIGURACIÓN Y CONEXIÓN AZURE SQL
# ============================================================

import os
import getpass
import secrets
import threading
import time
import traceback
import requests

import pandas as pd
import numpy as np

from datetime import datetime

from fastapi import FastAPI, HTTPException, Body, Header
from fastapi.responses import JSONResponse

from sklearn.ensemble import RandomForestClassifier

# ============================================================
# 1. IMPORTACIÓN DEL DRIVER MSSQL-PYTHON
# ============================================================

try:
    from mssql_python import connect

    print("✅ Driver mssql-python disponible")

except ImportError:
    raise ImportError(
        "No se encuentra el driver mssql-python.\n"
        "Ejecuta previamente:\n"
        "!pip install -q mssql-python"
    )

# ============================================================
# 2. CONFIGURACIÓN DE AZURE SQL
# ============================================================

SQL_SERVER = "nexusadmin.database.windows.net"
SQL_USER = "nexusadmin"

SQL_PASSWORD = getpass.getpass(
    "Ingresa la contraseña de Azure SQL: "
)

DB_OLTP = "NEXUS_OLTP"
DB_DW = "NEXUS"

# Validación de configuración
if not SQL_PASSWORD:
    raise ValueError("La contraseña de Azure SQL no puede estar vacía.")

print("✅ Configuración de Azure SQL definida")

# ============================================================
# 3. TOKEN DE AUTENTICACIÓN PHP - COLAB
# ============================================================

NEXUS_ETL_TOKEN = getpass.getpass(
    "Ingresa el token compartido entre PHP y Colab: "
).strip()

if not NEXUS_ETL_TOKEN:
    raise ValueError("El token de autenticación no puede estar vacío.")

if len(NEXUS_ETL_TOKEN) < 24:
    raise ValueError(
        "El token debe tener al menos 24 caracteres."
    )

print("✅ Token de autenticación validado")

# ============================================================
# 4. CONSULTAR IP PÚBLICA DE COLAB
# ============================================================

def obtener_ip_publica_colab():

    try:

        respuesta = requests.get(
            "https://api.ipify.org",
            timeout=15
        )

        respuesta.raise_for_status()

        ip = respuesta.text.strip()

        print("\n" + "=" * 60)
        print("IP PÚBLICA DEL ENTORNO COLAB")
        print("=" * 60)
        print("IP:", ip)
        print("\nDebes autorizar esta IP en Azure SQL:")
        print("Azure Portal > SQL Server > Networking")
        print("Firewall rules > Add client IP")
        print("=" * 60)

        return ip

    except requests.RequestException as error:

        print(
            "⚠️ No se pudo obtener la IP pública de Colab:",
            error
        )

        return None


COLAB_PUBLIC_IP = obtener_ip_publica_colab()

# ============================================================
# 5. FUNCIÓN DE CONEXIÓN A AZURE SQL
# ============================================================

def crear_conexion_sql(database):

    if not database:
        raise ValueError(
            "Debe especificarse el nombre de la base de datos."
        )

    connection_string = (
        f"Server={SQL_SERVER};"
        f"Database={database};"
        f"UID={SQL_USER};"
        f"PWD={SQL_PASSWORD};"
        "Encrypt=yes;"
        "TrustServerCertificate=no;"
    )

    try:

        conexion = connect(connection_string)

        return conexion

    except Exception as error:

        print(
            f"\n❌ Error al conectar con la base de datos {database}"
        )

        print("Tipo de error:", type(error).__name__)
        print("Detalle:", error)

        raise

# ============================================================
# 6. CONEXIÓN A NEXUS_OLTP
# ============================================================

print("\n" + "=" * 60)
print("CONECTANDO A NEXUS_OLTP")
print("=" * 60)

conn_oltp = crear_conexion_sql(DB_OLTP)

try:

    resultado_oltp = conn_oltp.execute(
        "SELECT DB_NAME()"
    ).fetchone()

    print(
        "✅ Conexión OLTP exitosa:",
        resultado_oltp[0]
    )

except Exception as error:

    print("❌ Error verificando la conexión OLTP:")
    print(error)

    raise

# ============================================================
# 7. CONEXIÓN A NEXUS DATA WAREHOUSE
# ============================================================

print("\n" + "=" * 60)
print("CONECTANDO A NEXUS DATA WAREHOUSE")
print("=" * 60)

conn_dw = crear_conexion_sql(DB_DW)

try:

    resultado_dw = conn_dw.execute(
        "SELECT DB_NAME()"
    ).fetchone()

    print(
        "✅ Conexión DW exitosa:",
        resultado_dw[0]
    )

except Exception as error:

    print("❌ Error verificando la conexión DW:")
    print(error)

    raise

# ============================================================
# 8. VARIABLES DE EXTRACCIÓN - CAPA SOURCE
# ============================================================

usuarios_src = None
interacciones_src = None
sesiones_src = None

# ============================================================
# 9. VARIABLES DE TRANSFORMACIÓN - CAPA STAGING
# ============================================================

usuarios_stg = None
interacciones_stg = None
sesiones_stg = None

# ============================================================
# 10. VARIABLES DE PREDICCIÓN Y ANALÍTICA
# ============================================================

predicciones = None
predictions_payload = None
kpi_df = None
riesgo_df = None

# ============================================================
# 11. VARIABLES PARA POWER BI
# ============================================================

usuarios_powerbi = None
interacciones_powerbi = None

# ============================================================
# 12. ESTADO DE LAS CAPAS ETL
# ============================================================

ETL_STATE = {

    "extract": None,
    "transform": None,
    "load": None,
    "predictions": None,
    "kpi": None,
    "bi": None

}

ETL_LAST_ERROR = None

ETL_BUSY = False

ETL_LOCK = threading.Lock()

# ============================================================
# 13. VALIDACIÓN FINAL
# ============================================================

print("\n")
print("=" * 60)
print("NEXUS BI - CONFIGURACIÓN COMPLETADA")
print("=" * 60)

print("Servidor Azure SQL :", SQL_SERVER)
print("Base OLTP         :", DB_OLTP)
print("Data Warehouse     :", DB_DW)
print("Driver            : mssql-python")

print("-" * 60)

print("Estado de conexiones:")

print("OLTP:", "🟢 CONECTADO")
print("DW  :", "🟢 CONECTADO")

print("-" * 60)

print("Capas ETL:")
print("Extract       : Preparado")
print("Staging       : Preparado")
print("Transform     : Preparado")
print("Load          : Preparado")
print("Predictions   : Preparado")
print("KPI           : Preparado")
print("BI            : Preparado")

print("=" * 60)
print("🚀 NEXUS BI LISTO PARA EJECUTAR EL PIPELINE")
print("=" * 60)

## 2. Capa 1 — Extract: fuentes OLTP


In [ ]:
# ============================================================
# NEXUS BI - EXTRACT CON MEDICIÓN Y VALIDACIÓN
# ============================================================
def capa_extract():
    global usuarios_src, interacciones_src, sesiones_src
    started = time.perf_counter()
    try:
        # Se leen únicamente una vez por ejecución del pipeline.
        # La extracción completa es necesaria para las capas de staging/predicción.
        usuarios_src = pd.read_sql_query("SELECT * FROM dbo.usuarios", conn_oltp)
        interacciones_src = pd.read_sql_query("SELECT * FROM dbo.interacciones", conn_oltp)
        sesiones_src = pd.read_sql_query("SELECT * FROM dbo.sesiones", conn_oltp)
        elapsed = round(time.perf_counter() - started, 2)
        processed = len(usuarios_src) + len(interacciones_src) + len(sesiones_src)
        resultado = {
            "status": "ok", "users": len(usuarios_src),
            "interactions": len(interacciones_src), "sessions": len(sesiones_src),
            "processed": processed, "observations": 0,
            "duration_seconds": elapsed, "at": datetime.now().isoformat()
        }
        ETL_STATE["extract"] = resultado
        print(f"EXTRACT | users={len(usuarios_src):,} interactions={len(interacciones_src):,} sessions={len(sesiones_src):,} total={processed:,} elapsed={elapsed}s")
        return resultado
    except Exception as e:
        ETL_STATE["extract"] = {"status":"error", "error":str(e), "duration_seconds":round(time.perf_counter()-started,2), "at":datetime.now().isoformat()}
        raise


## 3. Capa 2 — Staging Area


In [ ]:
# ============================================================
# NEXUS BI - STAGING OPTIMIZADO
# ============================================================
def capa_staging():
    global usuarios_stg, interacciones_stg, sesiones_stg
    if usuarios_src is None or interacciones_src is None or sesiones_src is None:
        capa_extract()
    usuarios_stg = usuarios_src.copy()
    interacciones_stg = interacciones_src.copy()
    sesiones_stg = sesiones_src.copy()
    for df in (usuarios_stg, interacciones_stg, sesiones_stg):
        df.columns = [str(c).strip().lower() for c in df.columns]
        if "id" in df.columns:
            df.drop_duplicates(subset=["id"], keep="last", inplace=True)
    if "dni" in usuarios_stg.columns:
        usuarios_stg.drop_duplicates(subset=["dni"], keep="last", inplace=True)
    if "riesgo" in usuarios_stg.columns:
        usuarios_stg["riesgo"] = pd.to_numeric(usuarios_stg["riesgo"], errors="coerce")
        usuarios_stg = usuarios_stg[usuarios_stg["riesgo"].between(1,5)].copy()
    ETL_STATE["staging"] = {
        "status":"ok", "users":len(usuarios_stg),
        "interactions":len(interacciones_stg), "sessions":len(sesiones_stg),
        "at":datetime.now().isoformat()
    }
    print(f"STAGING OK | Usuarios {len(usuarios_stg):,} | Interacciones {len(interacciones_stg):,} | Sesiones {len(sesiones_stg):,}")
    return ETL_STATE["staging"]


## 4. Capa 3 — Transformación ETL


In [ ]:

def capa_transform():
    global usuarios_stg, interacciones_stg, sesiones_stg

    if usuarios_stg is None:
        capa_staging()

    # ==========================================
    # CONVERTIR FECHAS
    # ==========================================

    date_configs = [
        (usuarios_stg, ["fecha_registro", "fecha_nacimiento"]),
        (interacciones_stg, ["fecha"]),
        (sesiones_stg, ["fecha"])
    ]

    invalid_dates = 0

    for df, date_cols in date_configs:
        for col in date_cols:
            if col in df.columns:
                original = df[col].copy()

                converted = pd.to_datetime(
                    original,
                    errors="coerce"
                )

                invalid_dates += (
                    original.notna() & converted.isna()
                ).sum()

                df[col] = converted


    # ==========================================
    # LIMPIAR TEXTO
    # ==========================================

    for df in (
        usuarios_stg,
        interacciones_stg,
        sesiones_stg
    ):
        for col in df.select_dtypes(include=["object", "string"]).columns:
            # Limpieza vectorizada de texto (eficiente para datasets grandes).
            df[col] = df[col].astype("string").str.strip()
            df[col] = df[col].replace({"": pd.NA})


    # ==========================================
    # CLASIFICACIÓN DE RIESGO
    # ==========================================

    if "riesgo" in usuarios_stg.columns:

        usuarios_stg["nivel_riesgo"] = (
            pd.to_numeric(
                usuarios_stg["riesgo"],
                errors="coerce"
            )
            .astype("Int64")
        )

        usuarios_stg["categoria_riesgo"] = np.select(
            [
                usuarios_stg["nivel_riesgo"].le(2),
                usuarios_stg["nivel_riesgo"].eq(3),
                usuarios_stg["nivel_riesgo"].ge(4)
            ],
            [
                "Bajo",
                "Medio",
                "Alto"
            ],
            default="No definido"
        )


    # ==========================================
    # DESGLOSE TEMPORAL INTERACCIONES
    # ==========================================

    if "fecha" in interacciones_stg.columns:

        interacciones_stg["anio"] = (
            interacciones_stg["fecha"].dt.year
        )

        interacciones_stg["mes"] = (
            interacciones_stg["fecha"].dt.month
        )

        interacciones_stg["dia"] = (
            interacciones_stg["fecha"].dt.day
        )


    # ==========================================
    # CONTADORES
    # ==========================================

    total_users = len(usuarios_stg)
    total_interactions = len(interacciones_stg)
    total_sessions = len(sesiones_stg)

    processed = (
        total_users
        + total_interactions
        + total_sessions
    )

    valid_users = (
        usuarios_stg["id"].notna().sum()
        if "id" in usuarios_stg.columns
        else 0
    )

    valid_interactions = (
        interacciones_stg["id"].notna().sum()
        if "id" in interacciones_stg.columns
        else 0
    )

    valid_sessions = (
        sesiones_stg["id"].notna().sum()
        if "id" in sesiones_stg.columns
        else 0
    )

    valid_total = (
        valid_users
        + valid_interactions
        + valid_sessions
    )

    observations = (
        processed - valid_total
    ) + invalid_dates


    # ==========================================
    # RESULTADO TRANSFORM
    # ==========================================

    resultado = {
        "status": "ok",
        "users": total_users,
        "interactions": total_interactions,
        "sessions": total_sessions,
        "processed": processed,
        "valid": int(valid_total),
        "observations": int(observations),
        "invalid_dates": int(invalid_dates),
        "at": datetime.now().isoformat()
    }

    ETL_STATE["transform"] = resultado

    print("========== ETL TRANSFORM ==========")
    print("Usuarios:", total_users)
    print("Interacciones:", total_interactions)
    print("Sesiones:", total_sessions)
    print("Procesados:", processed)
    print("Válidos:", valid_total)
    print("Observaciones:", observations)
    print("Fechas inválidas:", invalid_dates)
    print("===============================")

    return resultado


## 5. Capa 4 — Preparación de estructuras Data Warehouse


In [ ]:
def crear_estructuras_dw():
    ddl = [
    "IF OBJECT_ID('dbo.NX_DimUser','U') IS NULL CREATE TABLE dbo.NX_DimUser (UserKey INT IDENTITY(1,1) PRIMARY KEY, SourceUserID INT NOT NULL UNIQUE, DNI NVARCHAR(20) NULL, FullName NVARCHAR(200) NULL, Gender NVARCHAR(50) NULL, Email NVARCHAR(200) NULL, RiskLevel INT NULL, RiskCategory NVARCHAR(30) NULL, Status NVARCHAR(40) NULL, RegistrationDate DATE NULL, LocationKey INT NULL)",
    "IF OBJECT_ID('dbo.NX_DimLocation','U') IS NULL CREATE TABLE dbo.NX_DimLocation (LocationKey INT IDENTITY(1,1) PRIMARY KEY, Country NVARCHAR(80) NULL, Department NVARCHAR(80) NULL, Province NVARCHAR(80) NULL, District NVARCHAR(100) NULL, Address NVARCHAR(250) NULL)",
    "IF OBJECT_ID('dbo.NX_DimDevice','U') IS NULL CREATE TABLE dbo.NX_DimDevice (DeviceKey INT IDENTITY(1,1) PRIMARY KEY, Device NVARCHAR(80) NULL, OperatingSystem NVARCHAR(80) NULL)",
    "IF OBJECT_ID('dbo.NX_DimInteraction','U') IS NULL CREATE TABLE dbo.NX_DimInteraction (InteractionKey INT IDENTITY(1,1) PRIMARY KEY, InteractionType NVARCHAR(80) NULL, Category NVARCHAR(100) NULL, Format NVARCHAR(50) NULL)",
    "IF OBJECT_ID('dbo.NX_DimDate','U') IS NULL CREATE TABLE dbo.NX_DimDate (DateKey INT PRIMARY KEY, FullDate DATE NOT NULL UNIQUE, YearNum INT NULL, MonthNum INT NULL, DayNum INT NULL)",
    "IF OBJECT_ID('dbo.NX_FactInteraction','U') IS NULL CREATE TABLE dbo.NX_FactInteraction (FactKey BIGINT IDENTITY(1,1) PRIMARY KEY, SourceInteractionID INT NULL UNIQUE, UserKey INT NULL, DateKey INT NULL, DeviceKey INT NULL, InteractionKey INT NULL, DurationSeconds INT NULL, Views INT NULL)",
    "IF OBJECT_ID('dbo.NX_FactSession','U') IS NULL CREATE TABLE dbo.NX_FactSession (FactSessionKey BIGINT IDENTITY(1,1) PRIMARY KEY, SourceSessionID INT NULL UNIQUE, UserKey INT NULL, DateKey INT NULL, DurationMinutes DECIMAL(10,2) NULL, Views INT NULL)",
    "IF OBJECT_ID('dbo.NX_PredictionRisk','U') IS NULL CREATE TABLE dbo.NX_PredictionRisk (PredictionKey BIGINT IDENTITY(1,1) PRIMARY KEY, SourceUserID INT NOT NULL, PredictedRisk INT NOT NULL, RiskLabel NVARCHAR(30) NULL, ModelName NVARCHAR(100) NULL, RunAt DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME())",
    "IF OBJECT_ID('dbo.NX_ETLLog','U') IS NULL CREATE TABLE dbo.NX_ETLLog (RunID BIGINT IDENTITY(1,1) PRIMARY KEY, StartedAt DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME(), CompletedAt DATETIME2 NULL, SourceUsers INT NULL, SourceInteractions INT NULL, SourceSessions INT NULL, Status NVARCHAR(30) NULL, Details NVARCHAR(2000) NULL)"
    ]
    cur=conn_dw.cursor()
    for sql in ddl: cur.execute(sql)
    conn_dw.commit()
    print("Tablas NX_ verificadas/creadas")

def _safe(v):
    if pd.isna(v): return None
    if isinstance(v, pd.Timestamp): return v.to_pydatetime()
    return v

def _date_key(value):
    d=pd.to_datetime(value,errors="coerce")
    if pd.isna(d): return None, None
    return int(d.strftime("%Y%m%d")), d

def _ensure_date(cur, value):
    key,d=_date_key(value)
    if key is None: return None
    cur.execute("IF NOT EXISTS (SELECT 1 FROM dbo.NX_DimDate WHERE DateKey=?) INSERT dbo.NX_DimDate(DateKey,FullDate,YearNum,MonthNum,DayNum) VALUES(?,?,?,?,?)", (key,d.date(),d.year,d.month,d.day))
    return key

def _ensure_dim(cur, table, cols, vals):

    # ============================================
    # 1. BUSCAR SI LA DIMENSIÓN YA EXISTE
    # ============================================
    conditions = []
    params = []

    for col, val in zip(cols, vals):
        conditions.append(
            f"({col} = ? OR ({col} IS NULL AND ? IS NULL))"
        )
        params.extend([val, val])

    where = " AND ".join(conditions)

    cur.execute(
        f"""
        SELECT TOP 1 *
        FROM dbo.{table}
        WHERE {where}
        """,
        params
    )

    row = cur.fetchone()

    if row:
        return row[0]

    # ============================================
    # 2. INSERTAR Y RECUPERAR ID DIRECTAMENTE
    # ============================================
    colsql = ",".join(cols)
    marks = ",".join("?" for _ in vals)

    sql = f"""
        INSERT INTO dbo.{table} ({colsql})
        OUTPUT INSERTED.{cols[0] if table == 'NX_DimLocation' else 'InteractionKey' if table == 'NX_DimInteraction' else 'DeviceKey'}
        VALUES ({marks})
    """

    cur.execute(sql, vals)

    result = cur.fetchone()

    if result is None:
        raise Exception(
            f"No se pudo recuperar la clave generada para {table}"
        )

    return int(result[0])


## 6. Capa 4 — Load: carga incremental optimizada por lotes

Carga dimensiones y hechos con consultas agrupadas, caché de claves, detección de existentes por bloques y commits por lote.


In [ ]:
# ============================================================
# NEXUS BI - CAPA LOAD OPTIMIZADA
# ============================================================
# Características:
# - Inserción masiva con executemany
# - Caché de dimensiones
# - Resolución de claves en memoria
# - Actualización de usuarios existentes
# - Control de registros duplicados
# - Inserción de fechas
# - Carga de usuarios, interacciones y sesiones
# ============================================================

LOAD_BATCH_SIZE = 1000
SQL_IN_CHUNK = 900


# ============================================================
# 1. NORMALIZACIÓN DE CLAVES
# ============================================================

def _load_key(v):

    v = _safe(v)

    if isinstance(v, (pd.Timestamp, datetime)):
        return pd.Timestamp(v).isoformat()

    try:
        if isinstance(v, np.generic):
            return v.item()
    except Exception:
        pass

    return v


def _load_dim_tuple(vals):

    return tuple(_load_key(v) for v in vals)


# ============================================================
# 2. LECTURA DE DIMENSIONES EXISTENTES
# ============================================================

def _load_read_dim(cur, table, keycol, cols):

    cur.execute(
        f"SELECT {keycol}, {','.join(cols)} FROM dbo.{table}"
    )

    result = {}

    for row in cur.fetchall():

        result[_load_dim_tuple(row[1:])] = int(row[0])

    return result


# ============================================================
# 3. LECTURA MASIVA DE IDS
# ============================================================

def _load_read_ids(cur, table, keycol, sourcecol, ids):

    result = {}

    ids = list(
        dict.fromkeys(
            int(x)
            for x in ids
            if x is not None
        )
    )

    for start in range(0, len(ids), SQL_IN_CHUNK):

        chunk = ids[start:start + SQL_IN_CHUNK]

        if not chunk:
            continue

        marks = ",".join("?" for _ in chunk)

        cur.execute(
            f"""
            SELECT {sourcecol}, {keycol}
            FROM dbo.{table}
            WHERE {sourcecol} IN ({marks})
            """,
            chunk
        )

        result.update(
            {
                int(r[0]): int(r[1])
                for r in cur.fetchall()
            }
        )

    return result


# ============================================================
# 4. CARGA MASIVA DE FECHAS
# ============================================================

def _load_date_keys(cur, date_values):

    normalized = {}

    for value in date_values:

        key, d = _date_key(value)

        if key is not None:

            normalized[key] = d

    if not normalized:

        return {}

    keys = list(normalized)

    existing = set()

    for start in range(0, len(keys), SQL_IN_CHUNK):

        chunk = keys[start:start + SQL_IN_CHUNK]

        marks = ",".join("?" for _ in chunk)

        cur.execute(
            f"""
            SELECT DateKey
            FROM dbo.NX_DimDate
            WHERE DateKey IN ({marks})
            """,
            chunk
        )

        existing.update(
            int(r[0])
            for r in cur.fetchall()
        )

    missing = [
        k
        for k in keys
        if k not in existing
    ]

    if missing:

        cur.executemany(
            """
            INSERT INTO dbo.NX_DimDate
            (
                DateKey,
                FullDate,
                YearNum,
                MonthNum,
                DayNum
            )
            VALUES (?, ?, ?, ?, ?)
            """,
            [
                (
                    k,
                    normalized[k].date(),
                    int(normalized[k].year),
                    int(normalized[k].month),
                    int(normalized[k].day)
                )
                for k in missing
            ]
        )

    return {
        k: k
        for k in keys
    }


# ============================================================
# 5. RESOLVER MASIVO DE DIMENSIONES
# ============================================================

def _load_ensure_dims_batch(
    cur,
    table,
    keycol,
    cols,
    values_list
):

    unique_vals = list(
        dict.fromkeys(
            _load_dim_tuple(v)
            for v in values_list
        )
    )

    existing_map = _load_read_dim(
        cur,
        table,
        keycol,
        cols
    )

    missing = [
        v
        for v in unique_vals
        if v not in existing_map
    ]

    if missing:

        cur.executemany(
            f"""
            INSERT INTO dbo.{table}
            ({','.join(cols)})
            VALUES ({','.join('?' for _ in cols)})
            """,
            [
                tuple(_safe(x) for x in vals)
                for vals in missing
            ]
        )

        conn_dw.commit()

        existing_map = _load_read_dim(
            cur,
            table,
            keycol,
            cols
        )

    return {
        v: existing_map[v]
        for v in unique_vals
        if v in existing_map
    }


# ============================================================
# 6. CAPA LOAD PRINCIPAL
# ============================================================

def capa_load():

    global usuarios_stg
    global interacciones_stg
    global sesiones_stg

    cur = None

    started = time.perf_counter()

    try:

        # ----------------------------------------------------
        # VERIFICAR STAGING
        # ----------------------------------------------------

        if (
            usuarios_stg is None
            or interacciones_stg is None
            or sesiones_stg is None
        ):

            capa_transform()

        # ----------------------------------------------------
        # PREPARAR DATA WAREHOUSE
        # ----------------------------------------------------

        crear_estructuras_dw()

        cur = conn_dw.cursor()

        # Convertir DataFrames a registros una sola vez

        users = usuarios_stg.to_dict("records")

        interactions = interacciones_stg.to_dict("records")

        sessions = sesiones_stg.to_dict("records")

        # ----------------------------------------------------
        # VALIDAR IDS
        # ----------------------------------------------------

        valid_users = [
            u
            for u in users
            if pd.notna(u.get("id"))
        ]

        valid_interactions = [
            a
            for a in interactions
            if pd.notna(a.get("id"))
        ]

        valid_sessions = [
            s
            for s in sessions
            if pd.notna(s.get("id"))
        ]

        # ====================================================
        # 7. DIMENSIÓN LOCATION
        # ====================================================

        loc_cols_src = [
            "pais",
            "departamento",
            "provincia",
            "distrito",
            "domicilio"
        ]

        loc_cols_dw = [
            "Country",
            "Department",
            "Province",
            "District",
            "Address"
        ]

        loc_values = []

        user_location_tuple = {}

        for u in valid_users:

            if all(c in u for c in loc_cols_src):

                vals = tuple(
                    _safe(u.get(c))
                    for c in loc_cols_src
                )

                user_location_tuple[int(u["id"])] = (
                    _load_dim_tuple(vals)
                )

                loc_values.append(vals)

        loc_map = {}

        if loc_values:

            loc_map = _load_ensure_dims_batch(
                cur,
                "NX_DimLocation",
                "LocationKey",
                loc_cols_dw,
                loc_values
            )

        # ====================================================
        # 8. DIMENSIÓN INTERACTION
        # ====================================================

        interaction_cols_src = [
            "tipo",
            "categoria",
            "formato"
        ]

        interaction_cols_dw = [
            "InteractionType",
            "Category",
            "Format"
        ]

        interaction_values = [
            tuple(
                _safe(a.get(c))
                for c in interaction_cols_src
            )
            for a in valid_interactions
        ]

        interaction_map = {}

        if interaction_values:

            interaction_map = _load_ensure_dims_batch(
                cur,
                "NX_DimInteraction",
                "InteractionKey",
                interaction_cols_dw,
                interaction_values
            )

        # ====================================================
        # 9. DIMENSIÓN DEVICE
        # ====================================================

        device_cols_src = [
            "dispositivo",
            "sistema_operativo"
        ]

        device_cols_dw = [
            "Device",
            "OperatingSystem"
        ]

        device_values = [
            tuple(
                _safe(a.get(c))
                for c in device_cols_src
            )
            for a in valid_interactions
        ]

        device_map = {}

        if device_values:

            device_map = _load_ensure_dims_batch(
                cur,
                "NX_DimDevice",
                "DeviceKey",
                device_cols_dw,
                device_values
            )

        # ====================================================
        # 10. DIMENSIÓN DATE
        # ====================================================

        all_dates = [
            u.get("fecha_registro")
            for u in valid_users
        ]

        all_dates += [
            a.get("fecha")
            for a in valid_interactions
        ]

        all_dates += [
            s.get("fecha")
            for s in valid_sessions
        ]

        date_map = _load_date_keys(
            cur,
            all_dates
        )

        conn_dw.commit()

        # ====================================================
        # 11. CARGA DE USUARIOS
        # ====================================================

        user_ids = [
            int(u["id"])
            for u in valid_users
        ]

        user_existing = _load_read_ids(
            cur,
            "NX_DimUser",
            "UserKey",
            "SourceUserID",
            user_ids
        )

        user_insert_rows = []

        user_update_rows = []

        user_location_key = {}

        for u in valid_users:

            sid = int(u["id"])

            loc_tuple = user_location_tuple.get(sid)

            lk = (
                loc_map.get(loc_tuple)
                if loc_tuple is not None
                else None
            )

            user_location_key[sid] = lk

            # Nombre completo

            full_name = " ".join(
                str(u.get(c) or "")
                for c in ["nombre", "apellidos"]
            ).strip() or None

            # ------------------------------------------------
            # CORRECCIÓN RISKLEVEL Y RISKCATEGORY
            # ------------------------------------------------

            risk_level = _safe(
                u.get("riesgo")
            )

            risk_category = _safe(
                u.get("categoria_riesgo")
            )

            # Si el nivel no existe, utilizar categoría
            # como respaldo, sin pasar dos parámetros a _safe.

            if risk_level is None:

                risk_level = risk_category

            values = (
                _safe(u.get("dni")),
                full_name,
                _safe(u.get("genero")),
                _safe(u.get("correo")),
                risk_level,
                risk_category,
                _safe(u.get("estado")),
                _safe(u.get("fecha_registro")),
                lk
            )

            if sid in user_existing:

                user_update_rows.append(
                    values + (sid,)
                )

            else:

                user_insert_rows.append(
                    (sid,) + values
                )

        # ----------------------------------------------------
        # INSERT USERS
        # ----------------------------------------------------

        insert_user_sql = """
        INSERT INTO dbo.NX_DimUser
        (
            SourceUserID,
            DNI,
            FullName,
            Gender,
            Email,
            RiskLevel,
            RiskCategory,
            Status,
            RegistrationDate,
            LocationKey
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """

        # ----------------------------------------------------
        # UPDATE USERS
        # ----------------------------------------------------

        update_user_sql = """
        UPDATE dbo.NX_DimUser
        SET
            DNI=?,
            FullName=?,
            Gender=?,
            Email=?,
            RiskLevel=?,
            RiskCategory=?,
            Status=?,
            RegistrationDate=?,
            LocationKey=?
        WHERE SourceUserID=?
        """

        ui = 0

        ua = 0

        # Insertar usuarios

        for start in range(
            0,
            len(user_insert_rows),
            LOAD_BATCH_SIZE
        ):

            batch = user_insert_rows[
                start:start + LOAD_BATCH_SIZE
            ]

            cur.executemany(
                insert_user_sql,
                batch
            )

            ui += len(batch)

            conn_dw.commit()

        # Actualizar usuarios

        for start in range(
            0,
            len(user_update_rows),
            LOAD_BATCH_SIZE
        ):

            batch = user_update_rows[
                start:start + LOAD_BATCH_SIZE
            ]

            cur.executemany(
                update_user_sql,
                batch
            )

            ua += len(batch)

            conn_dw.commit()

        # ====================================================
        # 12. RECUPERAR USERKEY
        # ====================================================

        fact_user_ids = set()

        for a in valid_interactions:

            if pd.notna(a.get("usuario_id")):

                fact_user_ids.add(
                    int(a["usuario_id"])
                )

        for s in valid_sessions:

            if pd.notna(s.get("usuario_id")):

                fact_user_ids.add(
                    int(s["usuario_id"])
                )

        user_key_map = _load_read_ids(
            cur,
            "NX_DimUser",
            "UserKey",
            "SourceUserID",
            list(fact_user_ids)
        )

        conn_dw.commit()

        # ====================================================
        # 13. FACT INTERACTION
        # ====================================================

        interaction_ids = [
            int(a["id"])
            for a in valid_interactions
        ]

        fact_existing = _load_read_ids(
            cur,
            "NX_FactInteraction",
            "FactKey",
            "SourceInteractionID",
            interaction_ids
        )

        interaction_fact_rows = []

        for a in valid_interactions:

            sid = int(a["id"])

            # Evitar duplicados

            if sid in fact_existing:

                continue

            date_key, _ = _date_key(
                a.get("fecha")
            )

            if date_key is None:

                continue

            u_id = (
                int(a["usuario_id"])
                if pd.notna(a.get("usuario_id"))
                else None
            )

            user_key = (
                user_key_map.get(u_id)
                if u_id is not None
                else None
            )

            interaction_tuple = tuple(
                _safe(a.get(c))
                for c in interaction_cols_src
            )

            device_tuple = tuple(
                _safe(a.get(c))
                for c in device_cols_src
            )

            interaction_fact_rows.append(
                (
                    sid,
                    user_key,
                    date_key,
                    device_map.get(
                        _load_dim_tuple(device_tuple)
                    ),
                    interaction_map.get(
                        _load_dim_tuple(interaction_tuple)
                    ),
                    _safe(a.get("tiempo_segundos")),
                    _safe(a.get("vistas"))
                )
            )

        insert_fact_interaction = """
        INSERT INTO dbo.NX_FactInteraction
        (
            SourceInteractionID,
            UserKey,
            DateKey,
            DeviceKey,
            InteractionKey,
            DurationSeconds,
            Views
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """

        ii = 0

        for start in range(
            0,
            len(interaction_fact_rows),
            LOAD_BATCH_SIZE
        ):

            batch = interaction_fact_rows[
                start:start + LOAD_BATCH_SIZE
            ]

            cur.executemany(
                insert_fact_interaction,
                batch
            )

            ii += len(batch)

            conn_dw.commit()

        # ====================================================
        # 14. FACT SESSION
        # ====================================================

        session_ids = [
            int(s["id"])
            for s in valid_sessions
        ]

        session_existing = _load_read_ids(
            cur,
            "NX_FactSession",
            "FactSessionKey",
            "SourceSessionID",
            session_ids
        )

        session_fact_rows = []

        for s in valid_sessions:

            sid = int(s["id"])

            # Evitar duplicados

            if sid in session_existing:

                continue

            date_key, _ = _date_key(
                s.get("fecha")
            )

            if date_key is None:

                continue

            u_id = (
                int(s["usuario_id"])
                if pd.notna(s.get("usuario_id"))
                else None
            )

            session_fact_rows.append(
                (
                    sid,
                    user_key_map.get(u_id)
                    if u_id is not None
                    else None,
                    date_key,
                    _safe(s.get("duracion_min")),
                    _safe(s.get("vistas"))
                )
            )

        insert_fact_session = """
        INSERT INTO dbo.NX_FactSession
        (
            SourceSessionID,
            UserKey,
            DateKey,
            DurationMinutes,
            Views
        )
        VALUES (?, ?, ?, ?, ?)
        """

        si = 0

        for start in range(
            0,
            len(session_fact_rows),
            LOAD_BATCH_SIZE
        ):

            batch = session_fact_rows[
                start:start + LOAD_BATCH_SIZE
            ]

            cur.executemany(
                insert_fact_session,
                batch
            )

            si += len(batch)

            conn_dw.commit()

        # ====================================================
        # 15. REGISTRO DE EJECUCIÓN
        # ====================================================

        ETL_STATE["load"] = {
            "status": "ok",

            "users_source": len(usuarios_stg),
            "users_inserted": ui,
            "users_updated": ua,

            "interactions_source": len(interacciones_stg),
            "interactions_inserted": ii,

            "sessions_source": len(sesiones_stg),
            "sessions_inserted": si,

            "batch_size": LOAD_BATCH_SIZE,

            "duration_seconds": round(
                time.perf_counter() - started,
                2
            ),

            "at": datetime.now().isoformat()
        }

        print(
            f"LOAD OK | "
            f"Nuevos usuarios: {ui} | "
            f"Actualizados: {ua} | "
            f"Interacciones: {ii} | "
            f"Sesiones: {si} | "
            f"Tiempo: {ETL_STATE['load']['duration_seconds']}s"
        )

        return ETL_STATE["load"]

    # ========================================================
    # CONTROL DE ERRORES
    # ========================================================

    except Exception as e:

        try:

            conn_dw.rollback()

        except Exception:

            pass

        ETL_STATE["load"] = {
            "status": "error",
            "error": str(e),
            "duration_seconds": round(
                time.perf_counter() - started,
                2
            ),
            "at": datetime.now().isoformat()
        }

        print(
            "ERROR LOAD:",
            str(e)
        )

        traceback.print_exc()

        raise

    # ========================================================
    # CIERRE DE CURSOR
    # ========================================================

    finally:

        if cur is not None:

            try:

                cur.close()

            except Exception:

                pass


# ============================================================
# 16. AUXILIAR DE COMPATIBILIDAD: DIMENSIONES
# ============================================================

def _ensure_dim_cursor(
    cur,
    table,
    cols,
    vals
):

    keys = {
        "NX_DimLocation": "LocationKey",
        "NX_DimInteraction": "InteractionKey",
        "NX_DimDevice": "DeviceKey"
    }

    keycol = keys.get(table)

    if not keycol:

        raise ValueError(
            f"Dimensión no permitida: {table}"
        )

    vals = tuple(
        _safe(v)
        for v in vals
    )

    cache = globals().setdefault(
        "_DIM_CURSOR_CACHE",
        {}
    )

    cache_key = (
        table,
        _load_dim_tuple(vals)
    )

    if cache_key in cache:

        return cache[cache_key]

    cond = []

    params = []

    for col, val in zip(cols, vals):

        cond.append(
            f"({col}=? OR ({col} IS NULL AND ? IS NULL))"
        )

        params.extend(
            [val, val]
        )

    query = f"""
    SELECT TOP 1 {keycol}
    FROM dbo.{table}
    WHERE {' AND '.join(cond)}
    """

    cur.execute(
        query,
        params
    )

    row = cur.fetchone()

    if row:

        cache[cache_key] = int(row[0])

        return int(row[0])

    cur.execute(
        f"""
        INSERT dbo.{table}
        ({','.join(cols)})
        OUTPUT INSERTED.{keycol}
        VALUES ({','.join('?' for _ in vals)})
        """,
        vals
    )

    row = cur.fetchone()

    if not row:

        raise RuntimeError(
            f"No se obtuvo la clave de {table}"
        )

    cache[cache_key] = int(row[0])

    return int(row[0])


# ============================================================
# 17. AUXILIAR DE COMPATIBILIDAD: FECHAS
# ============================================================

def _date_key(value):

    d = pd.to_datetime(
        value,
        errors="coerce"
    )

    if pd.isna(d):

        return None, None

    return (
        int(d.strftime("%Y%m%d")),
        d
    )


def _ensure_date_cursor(cur, value):

    key, d = _date_key(value)

    if key is None:

        return None

    cache = globals().setdefault(
        "_DATE_CURSOR_CACHE",
        {}
    )

    if key in cache:

        return cache[key]

    cur.execute(
        """
        SELECT DateKey
        FROM dbo.NX_DimDate
        WHERE DateKey=?
        """,
        (key,)
    )

    row = cur.fetchone()

    if row:

        cache[key] = int(row[0])

        return int(row[0])

    cur.execute(
        """
        INSERT dbo.NX_DimDate
        (
            DateKey,
            FullDate,
            YearNum,
            MonthNum,
            DayNum
        )
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            key,
            d.date(),
            int(d.year),
            int(d.month),
            int(d.day)
        )
    )

    cache[key] = key

    return key

## 7. Capa 5 — Predicciones dinámicas y proyección a 30 días


In [ ]:
# ============================================================
# NEXUS - CAPA 5: ANALÍTICA Y PREDICCIONES DINÁMICAS
# ============================================================

def capa_predictions():
    """
    Calcula métricas dinámicas sobre los datos actuales de staging.
    Incluye cinco bloques analíticos y una proyección de actividad a 30 días.
    Las estimaciones son estadísticas/heurísticas; no se presentan como certeza.
    """
    global predicciones, predictions_payload

    if usuarios_stg is None or interacciones_stg is None or sesiones_stg is None:
        capa_transform()

    # Copias solo las columnas necesarias: reduce RAM en datasets grandes.
    user_cols = [c for c in ["id", "riesgo", "categoria_riesgo"] if c in usuarios_stg.columns]
    interaction_cols = [c for c in ["usuario_id", "fecha", "categoria", "tipo"] if c in interacciones_stg.columns]
    users = usuarios_stg[user_cols].copy()
    interactions = interacciones_stg[interaction_cols].copy()

    # ----------------------------
    # Validación y normalización
    # ----------------------------
    if "id" not in users.columns:
        raise ValueError("La tabla usuarios debe contener la columna id.")
    if "riesgo" not in users.columns:
        raise ValueError("La tabla usuarios debe contener la columna riesgo.")

    users["riesgo"] = pd.to_numeric(users["riesgo"], errors="coerce")
    users = users[users["id"].notna()].copy()
    users["riesgo"] = users["riesgo"].fillna(1).clip(1, 5)

    if "usuario_id" in interactions.columns:
        interactions["usuario_id"] = pd.to_numeric(interactions["usuario_id"], errors="coerce")
    else:
        interactions["usuario_id"] = np.nan

    if "fecha" in interactions.columns:
        interactions["fecha"] = pd.to_datetime(interactions["fecha"], errors="coerce")
    else:
        interactions["fecha"] = pd.NaT

    now = pd.Timestamp.now()
    last30_start = now - pd.Timedelta(days=30)
    prev30_start = now - pd.Timedelta(days=60)

    recent = interactions[interactions["fecha"].between(last30_start, now, inclusive="both")].copy()
    previous = interactions[interactions["fecha"].between(prev30_start, last30_start, inclusive="left")].copy()

    # ----------------------------
    # 1. Nivel de riesgo
    # ----------------------------
    total_users = int(len(users))
    high_risk_count = int(users["riesgo"].ge(4).sum())
    risk_pct = round((high_risk_count / total_users * 100), 2) if total_users else 0
    risk_distribution = []
    for level in range(1, 6):
        count = int(users["riesgo"].eq(level).sum())
        risk_distribution.append({
            "level": level, "users": count,
            "percentage": round(count / total_users * 100, 2) if total_users else 0
        })

    # ----------------------------
    # 2. Disminución de actividad
    # Comparación de interacción agregada de los últimos 30 días
    # contra los 30 días anteriores.
    # ----------------------------
    recent_total = int(len(recent))
    previous_total = int(len(previous))
    if previous_total > 0:
        activity_change_pct = round((recent_total - previous_total) / previous_total * 100, 2)
    elif recent_total > 0:
        activity_change_pct = 100.0
    else:
        activity_change_pct = 0.0

    activity_direction = (
        "increased" if activity_change_pct > 0
        else "decreased" if activity_change_pct < 0
        else "stable"
    )

    # ----------------------------
    # 3. Interacción con contenidos
    # ----------------------------
    category_col = "categoria" if "categoria" in interactions.columns else None
    type_col = "tipo" if "tipo" in interactions.columns else None
    content_summary = []
    if category_col and not recent.empty:
        category_counts = recent[category_col].fillna("Sin categoría").astype(str).value_counts()
        content_summary = [
            {"category": str(cat), "interactions": int(cnt)}
            for cat, cnt in category_counts.head(10).items()
        ]
    top_category = content_summary[0]["category"] if content_summary else "Sin datos"
    type_summary = []
    if type_col and not recent.empty:
        type_counts = recent[type_col].fillna("Sin tipo").astype(str).value_counts()
        type_summary = [{"type": str(t), "count": int(n)} for t, n in type_counts.head(10).items()]

    # ----------------------------
    # 4. Riesgo de baja actividad
    # Usuarios con 0 o 1 interacción en los últimos 30 días.
    # ----------------------------
    recent_counts = recent.groupby("usuario_id").size() if not recent.empty else pd.Series(dtype="int64")
    user_ids = pd.to_numeric(users["id"], errors="coerce")
    users["interactions_30d"] = user_ids.map(recent_counts).fillna(0).astype(int)
    low_activity_users = int(users["interactions_30d"].le(1).sum())
    low_activity_pct = round(low_activity_users / total_users * 100, 2) if total_users else 0

    # ----------------------------
    # 5. Proyección de actividad 30 días
    # Usa días recientes observados; si no hay historial, indica que
    # no hay base suficiente en vez de inventar un pronóstico.
    # ----------------------------
    forecast = []
    forecast_method = "daily_average_30d"
    if not interactions.empty and interactions["fecha"].notna().any():
        daily = (
            interactions.dropna(subset=["fecha"])
            .assign(day=lambda df: df["fecha"].dt.date)
            .groupby("day").size()
        )
        daily = daily[daily.index >= last30_start.date()]
        avg_daily = float(daily.mean()) if len(daily) else 0.0
        base_total = int(len(recent))
        forecast_total = max(0, int(round(avg_daily * 30)))
        for offset in range(1, 31):
            forecast.append({
                "day": (now.date() + pd.Timedelta(days=offset)).isoformat(),
                "projected_interactions": int(round(avg_daily * offset))
            })
        forecast_info = {
            "available": bool(len(daily)),
            "period_days": 30,
            "observed_interactions": base_total,
            "average_daily_interactions": round(avg_daily, 2),
            "projected_interactions_30d": forecast_total,
            "method": forecast_method,
            "daily": forecast
        }
    else:
        forecast_info = {
            "available": False,
            "period_days": 30,
            "observed_interactions": 0,
            "average_daily_interactions": 0,
            "projected_interactions_30d": None,
            "method": "insufficient_history",
            "daily": []
        }

    predictions_payload = {
        "status": "ok",
        "generated_at": now.isoformat(),
        "window_days": 30,
        "risk": {
            "users_total": total_users,
            "high_risk_users": high_risk_count,
            "high_risk_percentage": risk_pct,
            "distribution": risk_distribution
        },
        "activity_decrease": {
            "interactions_recent_30d": recent_total,
            "interactions_previous_30d": previous_total,
            "change_percentage": activity_change_pct,
            "direction": activity_direction
        },
        "content_interactions": {
            "top_category": top_category,
            "categories": content_summary,
            "types": type_summary,
            "recent_interactions_30d": recent_total
        },
        "low_activity_risk": {
            "users_low_activity": low_activity_users,
            "percentage": low_activity_pct,
            "definition": "0 o 1 interacción durante los últimos 30 días"
        },
        "activity_forecast_30d": forecast_info
    }

    # Tabla por usuario para uso analítico local, sin exponerla en el endpoint.
    predicciones = users[["id", "riesgo", "interactions_30d"]].copy()
    predicciones.rename(columns={
        "id": "source_user_id",
        "riesgo": "current_risk"
    }, inplace=True)

    ETL_STATE["predictions"] = {
        "status": "ok",
        "rows": total_users,
        "window_days": 30,
        "generated_at": now.isoformat(),
        "summary": {
            "high_risk_percentage": risk_pct,
            "activity_change_percentage": activity_change_pct,
            "low_activity_percentage": low_activity_pct,
            "projected_interactions_30d": forecast_info["projected_interactions_30d"]
        }
    }

    print("========== NEXUS PREDICTIONS ==========")
    print("Usuarios:", total_users)
    print("Riesgo alto:", high_risk_count, f"({risk_pct}%)")
    print("Cambio de actividad:", activity_change_pct, "%")
    print("Usuarios con baja actividad:", low_activity_users, f"({low_activity_pct}%)")
    print("Categoría principal:", top_category)
    print("Proyección 30 días:", forecast_info["projected_interactions_30d"])
    print("====================================")

    return ETL_STATE["predictions"]


## 8. Capa 6 — KPI


In [ ]:
# ============================================================
# NEXUS BI - KPI CONSOLIDADOS
# ============================================================
def capa_kpi():
    global kpi_df, riesgo_df
    query_kpi = """
    SELECT
        (SELECT COUNT_BIG(*) FROM dbo.NX_DimUser) AS TotalUsuariosDW,
        (SELECT COUNT_BIG(*) FROM dbo.NX_FactInteraction) AS TotalInteraccionesDW,
        (SELECT COUNT_BIG(*) FROM dbo.NX_FactSession) AS TotalSesionesDW,
        (SELECT COUNT_BIG(*) FROM dbo.NX_DimUser WHERE RiskLevel >= 4) AS UsuariosRiesgoAlto,
        (SELECT COUNT_BIG(*) FROM dbo.NX_DimUser WHERE RiskLevel >= 4) * 100.0 /
            NULLIF((SELECT COUNT_BIG(*) FROM dbo.NX_DimUser),0) AS PorcentajeRiesgoAlto,
        (SELECT AVG(CAST(DurationSeconds AS FLOAT)) FROM dbo.NX_FactInteraction) AS DuracionPromedioSegundos,
        (SELECT SUM(Views) FROM dbo.NX_FactInteraction) AS VistasTotales
    """
    kpi_df = pd.read_sql_query(query_kpi, conn_dw)
    riesgo_df = pd.read_sql_query(
        "SELECT RiskCategory, COUNT_BIG(*) AS Usuarios FROM dbo.NX_DimUser GROUP BY RiskCategory ORDER BY RiskCategory",
        conn_dw
    )
    ETL_STATE["kpi"] = {"status":"ok", "data":kpi_df.to_dict(orient="records"), "risk_breakdown":riesgo_df.to_dict(orient="records"), "at":datetime.now().isoformat()}
    display(kpi_df)
    display(riesgo_df)
    return ETL_STATE["kpi"]


## 9. Capa 7 — Salida semántica para Power BI


In [ ]:
# ============================================================
# NEXUS BI - SALIDA SEMÁNTICA / PREVIEWS LIMITADOS
# ============================================================
BI_PREVIEW_LIMIT = 1000

def capa_bi():
    global usuarios_powerbi, interacciones_powerbi
    # Conteos completos sin traer todas las filas al notebook.
    cur = conn_dw.cursor()
    try:
        cur.execute("SELECT COUNT_BIG(*) FROM dbo.NX_DimUser")
        total_users = int(cur.fetchone()[0])
        cur.execute("SELECT COUNT_BIG(*) FROM dbo.NX_FactInteraction")
        total_interactions = int(cur.fetchone()[0])
    finally:
        try: cur.close()
        except Exception: pass

    usuarios_powerbi = pd.read_sql_query(
        f"SELECT TOP ({BI_PREVIEW_LIMIT}) SourceUserID,DNI,FullName,Gender,Email,RiskLevel,RiskCategory,Status,RegistrationDate FROM dbo.NX_DimUser ORDER BY SourceUserID",
        conn_dw
    )
    interacciones_powerbi = pd.read_sql_query(
        f"SELECT TOP ({BI_PREVIEW_LIMIT}) f.SourceInteractionID,d.FullDate,u.SourceUserID,u.RiskCategory,i.InteractionType,i.Category,i.Format,dev.Device,dev.OperatingSystem,f.DurationSeconds,f.Views FROM dbo.NX_FactInteraction f LEFT JOIN dbo.NX_DimDate d ON f.DateKey=d.DateKey LEFT JOIN dbo.NX_DimUser u ON f.UserKey=u.UserKey LEFT JOIN dbo.NX_DimInteraction i ON f.InteractionKey=i.InteractionKey LEFT JOIN dbo.NX_DimDevice dev ON f.DeviceKey=dev.DeviceKey ORDER BY f.SourceInteractionID",
        conn_dw
    )
    ETL_STATE["bi"] = {
        "status":"ok", "users_rows":total_users, "interaction_rows":total_interactions,
        "preview_limit":BI_PREVIEW_LIMIT, "users_preview_rows":len(usuarios_powerbi),
        "interactions_preview_rows":len(interacciones_powerbi), "at":datetime.now().isoformat()
    }
    display(usuarios_powerbi.head())
    display(interacciones_powerbi.head())
    print(f"BI | Registros completos: usuarios={total_users:,}, interacciones={total_interactions:,}; previews limitados a {BI_PREVIEW_LIMIT:,}")
    return ETL_STATE["bi"]


## 10. Pipeline completo y registro ETL


In [ ]:
def ejecutar_pipeline_completo():
    global ETL_LAST_ERROR
    inicio=datetime.now()
    ETL_LAST_ERROR=None
    try:
        capa_extract(); capa_staging(); capa_transform(); capa_load(); capa_predictions(); capa_kpi(); capa_bi()
        cur=conn_dw.cursor()
        cur.execute("INSERT dbo.NX_ETLLog(SourceUsers,SourceInteractions,SourceSessions,Status,Details) VALUES(?,?,?,?,?)",(len(usuarios_src),len(interacciones_src),len(sesiones_src),"COMPLETED","Pipeline API: extract, staging, transform, load, predictions, KPI, BI"))
        conn_dw.commit()
        return {"status":"completed","duration_seconds":round((datetime.now()-inicio).total_seconds(),2),"layers":ETL_STATE.copy()}
    except Exception as e:
        ETL_LAST_ERROR=str(e)
        try:
            cur=conn_dw.cursor(); cur.execute("INSERT dbo.NX_ETLLog(SourceUsers,SourceInteractions,SourceSessions,Status,Details) VALUES(?,?,?,?,?)",(len(usuarios_src) if usuarios_src is not None else 0,len(interacciones_src) if interacciones_src is not None else 0,len(sesiones_src) if sesiones_src is not None else 0,"FAILED",str(e)[:1900])); conn_dw.commit()
        except Exception: pass
        raise


## 11. API FastAPI — Endpoints para PHP

Endpoints protegidos por token:
- `POST /generate-users` — genera usuarios ficticios directamente en Colab y los inserta por lotes en `NEXUS_OLTP`. Body: `{"cantidad": 1000, "ejecutar_etl": false}`.
- `POST /extract`
- `POST /transform`
- `POST /load`
- `POST /run` (pipeline completo)
- `GET /predictions` (resumen dinámico de Capa 5, horizonte de 30 días)
- `GET /health` (estado del servicio)

La generación no ejecuta el ETL automáticamente por defecto; se puede solicitar con `ejecutar_etl=true`, aunque para volúmenes grandes se recomienda separar generación y ETL para evitar tiempos de espera del cliente.


In [ ]:
# ============================================================
# NEXUS BI - API FASTAPI CON CONTROL DE CONCURRENCIA
# ============================================================

# ============================================================
# GENERADOR MASIVO DE USUARIOS FICTICIOS EN COLAB
# ============================================================
import random
import string
from datetime import timedelta

GEN_BATCH_SIZE = 500
GEN_MAX_USERS_PER_REQUEST = 20000

NOMBRES_FICTICIOS = [
    "Alex", "Andrea", "Camila", "Diego", "Elena", "Gabriel", "Lucia",
    "Mateo", "Valeria", "Sofia", "Daniel", "Maria", "Adrian", "Paula",
    "Javier", "Renata", "Carlos", "Diana", "Fernando", "Ariana",
    "Luis", "Noa", "Bruno", "Alessia", "Marco", "Nicole", "Samuel",
    "Thiago", "Mia", "Leonardo", "Eva", "Emilio", "Isabella", "Nicolas"
]
APELLIDOS_FICTICIOS = [
    "Garcia", "Lopez", "Torres", "Flores", "Diaz", "Rojas", "Castillo",
    "Vargas", "Mendoza", "Rivera", "Cruz", "Morales", "Reyes", "Herrera",
    "Paredes", "Salazar", "Medina", "Aguilar", "Ortiz", "Sanchez", "Romero"
]
DISTRITOS_FICTICIOS = [
    "Ventanilla", "Callao", "Bellavista", "La Perla", "La Punta", "Carmen de la Legua",
    "San Miguel", "Los Olivos", "Comas", "San Martin de Porres", "Lima", "Surco"
]

def _sql_user_table_metadata(cur):
    cur.execute("""
        SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE, COLUMN_DEFAULT,
               COLUMNPROPERTY(OBJECT_ID('dbo.usuarios'), COLUMN_NAME, 'IsIdentity') AS IsIdentity
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA='dbo' AND TABLE_NAME='usuarios'
        ORDER BY ORDINAL_POSITION
    """)
    return [dict(zip(["name","type","nullable","default","identity"], row)) for row in cur.fetchall()]

def _synthetic_user_row(metadata, index, batch_nonce):
    first = random.choice(NOMBRES_FICTICIOS)
    last1, last2 = random.choice(APELLIDOS_FICTICIOS), random.choice(APELLIDOS_FICTICIOS)
    full_email = f"demo{batch_nonce}{index}@example.invalid"
    dni = f"9{(batch_nonce + index) % 10000000:07d}"
    phone = f"900{(batch_nonce + index) % 10000000:07d}"
    risk = random.choices([1,2,3,4,5], weights=[30,25,20,15,10], k=1)[0]
    district = random.choice(DISTRITOS_FICTICIOS)
    today = datetime.now().date()
    regdate = today - timedelta(days=random.randint(0, 1095))
    values = {}
    aliases = {
        "nombre": first, "nombres": first, "name": first, "first_name": first,
        "apellidos": f"{last1} {last2}", "apellido": last1, "surname": last1, "last_name": f"{last1} {last2}",
        "dni": dni, "documento": dni, "document_number": dni,
        "correo": full_email, "email": full_email, "mail": full_email,
        "telefono": phone, "phone": phone, "celular": phone, "mobile": phone,
        "genero": random.choice(["Femenino", "Masculino", "No especificado"]),
        "gender": random.choice(["Femenino", "Masculino", "No especificado"]),
        "riesgo": risk, "risklevel": risk, "risk_level": risk,
        "categoria_riesgo": {1: "Bajo", 2: "Bajo", 3: "Medio", 4: "Alto", 5: "Alto"}[risk],
        "segmentacion": risk, "segmentacion_riesgo": risk, "segmentation": risk,
        "estado": random.choice(["Activo", "Activo", "Activo", "Inactivo"]), "status": "Activo",
        "fecha_registro": regdate, "registrationdate": regdate, "registration_date": regdate,
        "pais": "Peru", "country": "Peru",
        "departamento": "Callao", "department": "Callao",
        "provincia": "Callao", "province": "Callao",
        "distrito": district, "district": district,
        "domicilio": f"Av. Demo {random.randint(100, 9999)}", "address": f"Av. Demo {random.randint(100, 9999)}",
        "fecha_nacimiento": today - timedelta(days=random.randint(18*365, 65*365)),
    }
    for m in metadata:
        col = m["name"]
        normalized = col.lower()
        if normalized in ("id", "usuario_id", "user_id") or m["identity"]:
            continue
        if normalized in values:
            values[col] = values[normalized]
            continue
        # Unknown non-null columns: fail safely instead of guessing business rules.
        if str(m["nullable"]).upper() == "NO" and m["default"] is None:
            raise HTTPException(status_code=422, detail=f"Columna obligatoria sin mapeo en usuarios: {col}. Ajusta el generador a la estructura OLTP.")
    return values

def generar_usuarios_ficticios_oltp(cantidad):
    if cantidad < 1 or cantidad > GEN_MAX_USERS_PER_REQUEST:
        raise ValueError(f"La cantidad debe estar entre 1 y {GEN_MAX_USERS_PER_REQUEST}.")
    cur = conn_oltp.cursor()
    started = time.perf_counter()
    inserted = 0
    nonce = int(time.time() * 1000) % 100000000
    try:
        metadata = _sql_user_table_metadata(cur)
        rows = [_synthetic_user_row(metadata, i, nonce) for i in range(cantidad)]
        cols = list(rows[0].keys()) if rows else []
        if not cols:
            return {"status":"ok","inserted":0}
        sql = f"INSERT INTO dbo.usuarios ({','.join('['+c+']' for c in cols)}) VALUES ({','.join('?' for _ in cols)})"
        for start in range(0, len(rows), GEN_BATCH_SIZE):
            batch = rows[start:start+GEN_BATCH_SIZE]
            cur.executemany(sql, [tuple(r.get(c) for c in cols) for r in batch])
            conn_oltp.commit()
            inserted += len(batch)
        elapsed = round(time.perf_counter()-started, 2)
        return {"status":"ok","requested":cantidad,"inserted":inserted,"batch_size":GEN_BATCH_SIZE,"duration_seconds":elapsed,"generated_at":datetime.now().isoformat()}
    except Exception:
        try: conn_oltp.rollback()
        except Exception: pass
        raise
    finally:
        try: cur.close()
        except Exception: pass


GENERATION_JOBS = {}
GENERATION_JOBS_LOCK = threading.Lock()

def _generation_worker(job_id, cantidad):
    try:
        with GENERATION_JOBS_LOCK:
            GENERATION_JOBS[job_id]["status"] = "running"
            GENERATION_JOBS[job_id]["started_at"] = datetime.now().isoformat()
        result = generar_usuarios_ficticios_oltp(cantidad)
        with GENERATION_JOBS_LOCK:
            GENERATION_JOBS[job_id].update(result)
            GENERATION_JOBS[job_id]["status"] = "completed"
            GENERATION_JOBS[job_id]["completed_at"] = datetime.now().isoformat()
    except Exception as exc:
        with GENERATION_JOBS_LOCK:
            GENERATION_JOBS[job_id].update({
                "status": "failed", "error": str(exc),
                "completed_at": datetime.now().isoformat()
            })
    finally:
        global ETL_BUSY
        ETL_BUSY = False
        if ETL_LOCK.locked():
            try: ETL_LOCK.release()
            except RuntimeError: pass

app=FastAPI(title="NEXUS ETL API",version="1.2")

def require_token(authorization: str | None):
    expected="Bearer "+NEXUS_ETL_TOKEN
    if not authorization or not secrets.compare_digest(authorization.strip(), expected):
        raise HTTPException(status_code=401, detail="Token inválido o ausente")

@app.get("/")
def home():
    return {"service":"NEXUS ETL API", "status":"online", "engine":"Google Colab", "layers":7, "load_mode":"batch", "synthetic_generator":"/generate-users"}


@app.get("/health")
def health():
    return {"status":"online", "state":ETL_STATE, "busy":ETL_BUSY, "last_error":ETL_LAST_ERROR}


@app.get("/predictions")
def api_predictions(authorization: str | None = Header(default=None)):
    require_token(authorization)
    if ETL_BUSY:
        raise HTTPException(status_code=409, detail="ETL en ejecución; inténtalo cuando finalice")
    try:
        capa_extract(); capa_staging(); capa_transform(); capa_predictions()
        return predictions_payload
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error al calcular predicciones: {e}")



@app.post("/generate-users")
def api_generate_users(payload: dict = Body(...), authorization: str | None = Header(default=None)):
    global ETL_BUSY
    require_token(authorization)
    if not ETL_LOCK.acquire(blocking=False):
        raise HTTPException(status_code=409, detail="ETL o generación ya está ejecutándose")
    cantidad = int(payload.get("cantidad", 0))
    if cantidad < 1 or cantidad > GEN_MAX_USERS_PER_REQUEST:
        ETL_LOCK.release()
        raise HTTPException(status_code=422, detail=f"La cantidad debe estar entre 1 y {GEN_MAX_USERS_PER_REQUEST}.")
    ETL_BUSY = True
    job_id = secrets.token_urlsafe(12)
    with GENERATION_JOBS_LOCK:
        GENERATION_JOBS[job_id] = {
            "job_id": job_id, "status": "queued", "requested": cantidad,
            "created_at": datetime.now().isoformat()
        }
    worker = threading.Thread(target=_generation_worker, args=(job_id, cantidad), daemon=True)
    worker.start()
    return {"status": "accepted", "job_id": job_id, "requested": cantidad,
            "status_endpoint": f"/generation-status?job_id={job_id}",
            "message": "Generación iniciada en Colab; consulta el estado usando job_id."}

@app.get("/generation-status")
def api_generation_status(job_id: str, authorization: str | None = Header(default=None)):
    require_token(authorization)
    with GENERATION_JOBS_LOCK:
        job = GENERATION_JOBS.get(job_id)
        if job is None:
            raise HTTPException(status_code=404, detail="Trabajo no encontrado")
        return dict(job)

@app.post("/extract")
def api_extract(authorization: str | None=Header(default=None)):
    require_token(authorization)
    if ETL_BUSY:
        raise HTTPException(status_code=409, detail="ETL en ejecución")
    try:
        return {"status":"ok", "layer":"extract", "result":capa_extract()}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/transform")
def api_transform(authorization: str | None=Header(default=None)):
    require_token(authorization)
    if ETL_BUSY:
        raise HTTPException(status_code=409, detail="ETL en ejecución")
    try:
        if usuarios_src is None:
            capa_extract()
        capa_staging()
        return {"status":"ok", "layer":"transform", "result":capa_transform()}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/load")
def api_load(authorization: str | None=Header(default=None)):
    global ETL_BUSY
    require_token(authorization)
    if not ETL_LOCK.acquire(blocking=False):
        raise HTTPException(status_code=409, detail="ETL ya está ejecutándose")
    ETL_BUSY=True
    try:
        if usuarios_stg is None or interacciones_stg is None or sesiones_stg is None:
            capa_transform()
        result=capa_load()
        pred=capa_predictions()
        kpi=capa_kpi()
        bi=capa_bi()
        return {"status":"ok", "layer":"load", "results":{"load":result, "predictions":pred, "kpi":kpi, "bi":bi}}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        ETL_BUSY=False
        ETL_LOCK.release()


@app.post("/run")
def api_run(authorization: str | None=Header(default=None)):
    global ETL_BUSY
    require_token(authorization)
    if not ETL_LOCK.acquire(blocking=False):
        raise HTTPException(status_code=409, detail="ETL ya está ejecutándose")
    ETL_BUSY=True
    try:
        return ejecutar_pipeline_completo()
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"ETL falló: {e}")
    finally:
        ETL_BUSY=False
        ETL_LOCK.release()


## 12. Iniciar API


In [ ]:
# ============================================================
# NEXUS BI - INICIAR API FASTAPI EN GOOGLE COLAB
# ============================================================

import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

# ============================================================
# CONFIGURACIÓN DEL SERVIDOR
# ============================================================

HOST = "0.0.0.0"
PORT = 8000

# ============================================================
# FUNCIÓN PARA EJECUTAR UVICORN
# ============================================================

def iniciar_servidor_nexus():

    config = uvicorn.Config(
        app=app,
        host=HOST,
        port=PORT,
        log_level="info"
    )

    servidor = uvicorn.Server(config)

    servidor.run()

# ============================================================
# INICIAR SERVIDOR EN SEGUNDO PLANO
# ============================================================

if "servidor_nexus_thread" not in globals() or not servidor_nexus_thread.is_alive():

    servidor_nexus_thread = threading.Thread(
        target=iniciar_servidor_nexus,
        daemon=True
    )

    servidor_nexus_thread.start()

    print("=" * 60)
    print("🚀 NEXUS BI API")
    print("=" * 60)
    print("Servidor iniciado en segundo plano")
    print(f"Host: {HOST}")
    print(f"Puerto: {PORT}")
    print("=" * 60)

else:

    print("⚠️ El servidor NEXUS BI ya está ejecutándose")

## 13. Crear túnel Cloudflare


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8000

## 14. Configuración PHP


Copia la URL pública de Cloudflare que genera la última celda y configura `config/etl.php` de XAMPP con esa URL y **el mismo token** que ingresaste en la celda de configuración. PHP envía `Authorization: Bearer TU_TOKEN`.

Endpoints principales:
- `POST /run`: pipeline completo.
- `POST /load`: carga, predicciones, KPI y previews.
- `POST /generate-users`: generación ficticia en Colab. Body JSON de ejemplo: `{"cantidad": 5000, "ejecutar_etl": false}`.

El endpoint de generación inicia un trabajo en segundo plano, inserta usuarios en `NEXUS_OLTP` por lotes de 500 y devuelve un `job_id`; el visual puede consultar `/generation-status` para mostrar avance/estado. Después ejecuta `/run` para actualizar el DW. La API impide ejecuciones simultáneas. El enlace Cloudflare `trycloudflare.com` es temporal y cambiará al reiniciar el runtime. Mantén Colab ejecutándose mientras la web utiliza el ETL.


## 15. Cerrar conexiones (opcional)


In [ ]:
for c in [conn_oltp,conn_dw]:
    try: c.close()
    except Exception: pass
print("Conexiones cerradas.")